# Computing Perplexity with GPT-2 and Qwen 1B

In this notebook, we walk through how to compute the **perplexity** of a language model on a given text.

Perplexity measures how "surprised" a model is by the text. Lower perplexity means the model assigns higher probability to the text.

$$\text{PPL}(W) = \exp\left(-\frac{1}{N}\sum_{i=1}^{N} \log P(w_i \mid w_{<i})\right)$$

where $N$ is the number of tokens, and $P(w_i \mid w_{<i})$ is the model's predicted probability for token $w_i$ given all preceding tokens.

## Step 1: Setup and Imports

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np

## Step 2: Define the Input Text

We will use a paragraph about the early history of the University of Chicago.

In [ ]:
text = (
    "William Rainey Harper became the university's president on July 1, 1891, "
    "and classes first began on October 1, 1892. Harper offered large salaries "
    "to attract senior faculty, and in two years had a faculty of 120, including "
    "eight former university or college presidents. The undergraduate program was "
    "divided into two parts, with the first two years making up the Academic College, "
    "focusing on preparation for higher learning, and the last two years comprising "
    "the University College, with more advanced courses."
)
print(text)

## Step 3: Load GPT-2

GPT-2 (124M parameters) was released by OpenAI in 2019. Let's load the model and tokenizer.

In [ ]:
gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")
gpt2_model = AutoModelForCausalLM.from_pretrained("gpt2")
gpt2_model.eval()  # set to evaluation mode (disables dropout)
print(f"GPT-2 parameters: {sum(p.numel() for p in gpt2_model.parameters()):,}")

## Step 4: Tokenize the Text

Before we can feed text to a model, we need to convert it into token IDs. Let's see what tokens GPT-2 produces.

In [ ]:
gpt2_inputs = gpt2_tokenizer(text, return_tensors="pt")
gpt2_input_ids = gpt2_inputs["input_ids"]

print(f"Number of tokens: {gpt2_input_ids.shape[1]}")
print()

# Show each token and its ID
tokens = gpt2_input_ids[0].tolist()
print("Token ID -> Token string:")
for i, tok_id in enumerate(tokens):
    tok_str = gpt2_tokenizer.decode([tok_id])
    print(f"  [{i:2d}] {tok_id:6d} -> {repr(tok_str)}")

## Step 5: Get Model Logits (Raw Predictions)

A causal language model outputs **logits** for each position: a vector of scores over the entire vocabulary.

For position $i$, the logits predict the distribution over the **next** token $w_{i+1}$.

So `logits[0]` predicts what comes after the first token, `logits[1]` predicts what comes after the second token, etc.

In [ ]:
with torch.no_grad():
    gpt2_outputs = gpt2_model(**gpt2_inputs)

gpt2_logits = gpt2_outputs.logits  # shape: (batch=1, seq_len, vocab_size)
print(f"Logits shape: {gpt2_logits.shape}")
print(f"Vocabulary size: {gpt2_logits.shape[-1]}")
print(f"Sequence length: {gpt2_logits.shape[1]}")

## Step 6: Convert Logits to Probabilities

We apply **softmax** to convert logits into a probability distribution:

$$P(w \mid \text{context}) = \frac{\exp(\text{logit}_w)}{\sum_{w'} \exp(\text{logit}_{w'})}$$

Then we extract the probability assigned to the **actual next token** at each position.

In [ ]:
# logits[:, :-1, :] gives predictions for positions 0..N-2 (predicting tokens 1..N-1)
# input_ids[:, 1:] gives the actual tokens at positions 1..N-1 (the targets)
shift_logits = gpt2_logits[:, :-1, :]  # predictions
shift_labels = gpt2_input_ids[:, 1:]    # targets

print(f"Predictions shape: {shift_logits.shape}  (predicting {shift_logits.shape[1]} tokens)")
print(f"Targets shape:     {shift_labels.shape}")

In [ ]:
# Convert logits to log-probabilities
log_probs = F.log_softmax(shift_logits, dim=-1)  # shape: (1, seq_len-1, vocab_size)

# Gather the log-probability of the actual next token at each position
# shift_labels has shape (1, seq_len-1), we need (1, seq_len-1, 1) for gather
token_log_probs = log_probs.gather(dim=-1, index=shift_labels.unsqueeze(-1)).squeeze(-1)

print(f"Per-token log probabilities shape: {token_log_probs.shape}")
print(f"\nFirst 10 token log-probs: {token_log_probs[0, :10].tolist()}")

## Step 7: Inspect Per-Token Surprisal

**Surprisal** (negative log-probability) tells us how "surprised" the model is at each token. Higher surprisal means the model found that token less predictable.

$$\text{surprisal}(w_i) = -\log P(w_i \mid w_{<i})$$

In [ ]:
# Show per-token surprisal
surprisals = -token_log_probs[0].tolist()

print(f"{'Position':>8}  {'Token':>15}  {'Log-Prob':>10}  {'Surprisal':>10}  {'Prob':>10}")
print("-" * 65)
for i, (tok_id, surp) in enumerate(zip(tokens[1:], surprisals)):
    tok_str = gpt2_tokenizer.decode([tok_id])
    log_p = -surp
    prob = np.exp(log_p)
    print(f"{i+1:>8}  {repr(tok_str):>15}  {log_p:>10.4f}  {surp:>10.4f}  {prob:>10.6f}")

## Step 8: Compute Perplexity for GPT-2

Perplexity is the exponential of the average negative log-likelihood:

$$\text{PPL} = \exp\left(-\frac{1}{N}\sum_{i=1}^{N} \log P(w_i \mid w_{<i})\right) = \exp(\text{average surprisal})$$

In [ ]:
# Average negative log-likelihood
avg_nll = -token_log_probs[0].mean().item()

# Perplexity = exp(average NLL)
gpt2_ppl = np.exp(avg_nll)

print(f"GPT-2 Results:")
print(f"  Number of tokens scored: {token_log_probs.shape[1]}")
print(f"  Total NLL:              {-token_log_probs[0].sum().item():.4f}")
print(f"  Average NLL:            {avg_nll:.4f}")
print(f"  Perplexity:             {gpt2_ppl:.2f}")

## Step 9: Load Qwen 1B

Now let's compare with a larger, more recent model: Qwen2.5-0.5B (released by Alibaba). It has roughly 0.5B parameters and was trained on more data than GPT-2.

We expect it to achieve lower perplexity (less surprised by the text).

In [ ]:
qwen_model_name = "Qwen/Qwen2.5-0.5B"

qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_model_name)
qwen_model = AutoModelForCausalLM.from_pretrained(qwen_model_name)
qwen_model.eval()
print(f"Qwen parameters: {sum(p.numel() for p in qwen_model.parameters()):,}")

## Step 10: Tokenize with Qwen's Tokenizer

Different models use different tokenizers. Let's compare how Qwen tokenizes the same text.

In [ ]:
qwen_inputs = qwen_tokenizer(text, return_tensors="pt")
qwen_input_ids = qwen_inputs["input_ids"]

print(f"GPT-2 tokens: {gpt2_input_ids.shape[1]}")
print(f"Qwen tokens:  {qwen_input_ids.shape[1]}")
print()

# Show Qwen's tokenization
qwen_tokens = qwen_input_ids[0].tolist()
print("Qwen Token ID -> Token string:")
for i, tok_id in enumerate(qwen_tokens):
    tok_str = qwen_tokenizer.decode([tok_id])
    print(f"  [{i:2d}] {tok_id:6d} -> {repr(tok_str)}")

## Step 11: Compute Qwen's Perplexity

We repeat the same process: get logits, extract log-probabilities for the actual tokens, and compute perplexity.

In [ ]:
with torch.no_grad():
    qwen_outputs = qwen_model(**qwen_inputs)

qwen_logits = qwen_outputs.logits
print(f"Qwen logits shape: {qwen_logits.shape}")
print(f"Qwen vocabulary size: {qwen_logits.shape[-1]}")

In [ ]:
# Same shifting: predictions for tokens 1..N-1
qwen_shift_logits = qwen_logits[:, :-1, :]
qwen_shift_labels = qwen_input_ids[:, 1:]

# Log-probabilities of actual next tokens
qwen_log_probs = F.log_softmax(qwen_shift_logits, dim=-1)
qwen_token_log_probs = qwen_log_probs.gather(
    dim=-1, index=qwen_shift_labels.unsqueeze(-1)
).squeeze(-1)

# Per-token surprisal for Qwen
qwen_surprisals = -qwen_token_log_probs[0].tolist()

print(f"{'Position':>8}  {'Token':>15}  {'Log-Prob':>10}  {'Surprisal':>10}  {'Prob':>10}")
print("-" * 65)
for i, (tok_id, surp) in enumerate(zip(qwen_tokens[1:], qwen_surprisals)):
    tok_str = qwen_tokenizer.decode([tok_id])
    log_p = -surp
    prob = np.exp(log_p)
    print(f"{i+1:>8}  {repr(tok_str):>15}  {log_p:>10.4f}  {surp:>10.4f}  {prob:>10.6f}")

In [ ]:
# Compute Qwen's perplexity
qwen_avg_nll = -qwen_token_log_probs[0].mean().item()
qwen_ppl = np.exp(qwen_avg_nll)

print(f"Qwen Results:")
print(f"  Number of tokens scored: {qwen_token_log_probs.shape[1]}")
print(f"  Total NLL:              {-qwen_token_log_probs[0].sum().item():.4f}")
print(f"  Average NLL:            {qwen_avg_nll:.4f}")
print(f"  Perplexity:             {qwen_ppl:.2f}")

## Step 12: Compare the Two Models

In [ ]:
print(f"{'Model':<15} {'Params':>12} {'Vocab Size':>12} {'Num Tokens':>12} {'Avg NLL':>10} {'Perplexity':>12}")
print("-" * 75)
print(f"{'GPT-2':<15} {'124M':>12} {gpt2_logits.shape[-1]:>12,} {token_log_probs.shape[1]:>12} {avg_nll:>10.4f} {gpt2_ppl:>12.2f}")
print(f"{'Qwen2.5-0.5B':<15} {'0.5B':>12} {qwen_logits.shape[-1]:>12,} {qwen_token_log_probs.shape[1]:>12} {qwen_avg_nll:>10.4f} {qwen_ppl:>12.2f}")

## Step 13: Reusable Perplexity Function

Let's package everything into a clean function.

In [ ]:
def compute_perplexity(model, tokenizer, text):
    """Compute perplexity of a causal LM on a given text."""
    inputs = tokenizer(text, return_tensors="pt")
    input_ids = inputs["input_ids"]

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits

    # Shift: logits[:-1] predict targets[1:]
    shift_logits = logits[:, :-1, :]
    shift_labels = input_ids[:, 1:]

    # Cross-entropy loss per token
    log_probs = F.log_softmax(shift_logits, dim=-1)
    token_log_probs = log_probs.gather(dim=-1, index=shift_labels.unsqueeze(-1)).squeeze(-1)

    avg_nll = -token_log_probs.mean().item()
    ppl = np.exp(avg_nll)

    return {
        "perplexity": ppl,
        "avg_nll": avg_nll,
        "num_tokens": token_log_probs.shape[1],
        "token_log_probs": token_log_probs[0].tolist(),
    }

In [ ]:
# Verify our function gives the same results
gpt2_result = compute_perplexity(gpt2_model, gpt2_tokenizer, text)
qwen_result = compute_perplexity(qwen_model, qwen_tokenizer, text)

print(f"GPT-2 perplexity:  {gpt2_result['perplexity']:.2f}")
print(f"Qwen perplexity:   {qwen_result['perplexity']:.2f}")

## Step 14: Experiment - Try Your Own Text

Try computing perplexity on different types of text. What do you expect?
- Common English sentences (low perplexity)
- Rare or technical text (higher perplexity)
- Random characters (very high perplexity)

In [ ]:
test_texts = [
    "The cat sat on the mat.",
    "The mitochondria is the powerhouse of the cell.",
    "Colorless green ideas sleep furiously.",
    "asdf jkl qwerty zxcv bnm poiu",
]

print(f"{'Text':<55} {'GPT-2 PPL':>10} {'Qwen PPL':>10}")
print("-" * 80)
for t in test_texts:
    g = compute_perplexity(gpt2_model, gpt2_tokenizer, t)
    q = compute_perplexity(qwen_model, qwen_tokenizer, t)
    display_text = t[:52] + "..." if len(t) > 55 else t
    print(f"{display_text:<55} {g['perplexity']:>10.2f} {q['perplexity']:>10.2f}")

## Key Takeaways

1. **Perplexity** measures how well a language model predicts a given text. Lower is better.
2. The computation involves: tokenize -> get logits -> softmax -> extract target token probabilities -> average NLL -> exponentiate.
3. The **shift by one** alignment is critical: logits at position $i$ predict the token at position $i+1$.
4. Different tokenizers produce different token counts, so comparing raw NLL totals across models is not meaningful. Perplexity (average per token) is comparable but keep in mind the tokenization difference.
5. Larger models trained on more data generally achieve lower perplexity.